# ITC C508 — Lab Exercise PT-M1

**Made By: BERNARDO, Luke Joaquin**

---


## Building and Evaluating a Domain-Specific RAG Chatbot



**Domain:** Monster Hunter World: Iceborne Lore Assistant

**Domain description:** This chatbot answers questions strictly grounded in monster, location, and story/faction lore from both the base game *Monster Hunter: World* and its expansion *Iceborne*.


**Note on the code below:** Cells 1–7 follow the lab's provided template exactly, as instructed. The only intentional change is the wording of the system prompt in Cell 5, since the template explicitly marks that line as the "Custom domain system prompt" to be filled in per student.

---

### Cell 1 — Environment Setup & Package Installation

This cell installs every library the RAG pipeline depends on, in one shot, before any other cell runs:

- **langchain** — the core framework that chains together loading, retrieval, and generation steps.
- **langchain-classic** — as of LangChain v1.0, several pre-built chain constructors (including `create_retrieval_chain` and `create_stuff_documents_chain`, used in Cell 6) were split out of the main `langchain` package into this companion package. It's required for the lab template's Cell 6 code to work on current LangChain versions.
- **langchain-community** — community-maintained integrations, including the `DirectoryLoader` used later to read files from `./my_data/`.
- **langchain-groq** — the connector that lets LangChain call Groq's hosted LLMs (we use `openai/gpt-oss-20b`).
- **langchain-huggingface** — lets LangChain use a local/open-source HuggingFace embedding model instead of a paid API.
- **chromadb** — the vector database that stores document embeddings and performs similarity search.
- **pypdf** — a PDF parser some loaders use; not strictly needed for our `.txt`-only dataset, but kept as-is per the provided template.
- **unstructured** — a fallback parser for messier file formats, used by some document loaders.

The `-q` flag just keeps the install output quiet, and `-U` forces pip to upgrade/reinstall even if it thinks a package is already present — this avoids stale or partial installs left over from a previous session, which is the most common cause of a later `ModuleNotFoundError` (e.g. `No module named 'langchain.chains'`) in Cell 6.


In [ ]:
# Cell 1: Environment Setup & Package Installation
!pip install -q -U langchain langchain-classic langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 49.0 MB/s eta 0:00:00
   

### Cell 2 — Secure API Key Configuration

This cell asks for the Groq API key at runtime instead of hardcoding it in the notebook, which is important because:

- `import os` gives access to environment variables, which is where LangChain expects to find `GROQ_API_KEY`.
- `import getpass` provides a prompt that does **not** echo the typed characters back to the screen, so the key isn't visible or accidentally saved in cell output.
- `if "GROQ_API_KEY" not in os.environ:` checks whether the key was already set in this session, so re-running the cell doesn't force you to re-enter it.
- `os.environ["GROQ_API_KEY"] = getpass.getpass(...)` stores the entered key in the environment for the rest of the session, where `ChatGroq` (used in Cell 5) will automatically look for it.

This keeps the key out of the notebook file itself, which matters if the `.ipynb` is later shared or submitted.

In [ ]:
# Cell 2: Secure API Key Configuration
import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


### Cell 3 — Loading Custom Data & Chunking

This cell reads the 3 lore `.txt` files (already uploaded to `./my_data/`, per the note above) and breaks them into small, searchable pieces:

- `from langchain_community.document_loaders import DirectoryLoader` — imports a loader that can read every file in a folder, rather than opening files one by one.
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` — imports the tool that will cut long documents into smaller chunks.
- `os.makedirs("my_data", exist_ok=True)` — ensures the target folder exists (harmless if the files were already uploaded there).
- `loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` — sets up the loader to read **every** file in `my_data/` (the `**/*.*` glob pattern), showing a progress bar as it works.
- `raw_documents = loader.load()` — actually reads the 3 lore files into memory as LangChain `Document` objects.
- `text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — configures the splitter to cut text into ~500-character chunks, with 50 characters of overlap between consecutive chunks so a sentence split across a chunk boundary isn't completely lost from either piece.
- `documents = text_splitter.split_documents(raw_documents)` — applies that splitting to all 3 files, producing many small chunks (one per monster/location/story entry, roughly) instead of three giant blobs of text.

Smaller chunks matter for RAG because the retriever (Cell 4) searches for the most *relevant* chunk to a query — e.g. a question about Velkhana should retrieve the Velkhana chunk specifically, not the entire monster-lore file.

In [ ]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory
loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 3/3 [00:18<00:00,  6.00s/it]

Loaded 3 raw document(s) and split into 787 chunks.


### Cell 4 — Embedding Model & Vector DB Indexing

This cell converts each text chunk into a numeric vector (embedding) and stores those vectors so they can be searched later:

- `from langchain_huggingface import HuggingFaceEmbeddings` — imports the wrapper that runs an open-source embedding model locally instead of calling a paid API.
- `from langchain_community.vectorstores import Chroma` — imports the Chroma vector database integration.
- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — loads a small, fast sentence-embedding model that converts text into 384-dimensional vectors capturing its meaning.
- `vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` — runs every chunk from Cell 3 through the embedding model and stores the resulting vectors in an in-memory ChromaDB instance, indexed for similarity search.
- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — wraps the vector store as a "retriever" object that, given a query, returns the **top 3** most similar chunks (`k=3`). This retriever is what Cell 6 plugs into the RAG chain.

This is the step that actually makes the chatbot "domain-specific" — without it, the LLM would have no way to find the relevant Monster Hunter lore inside our 3 files.

In [ ]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Cell 5 — Model Initialization and Domain System Prompt

This cell sets up the actual language model and tells it exactly how to behave for our domain:

- `from langchain_groq import ChatGroq` — imports the wrapper that connects to Groq's hosted inference API.
- `from langchain_core.prompts import ChatPromptTemplate` — imports the tool for building structured, multi-role prompts (system + human messages).
- `llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0)` — initializes the model. `temperature=0` makes output as deterministic/grounded as possible — this is our **baseline** configuration for Tasks 1 and 2.
- `system_prompt = (...)` — this is the domain guardrail, and the only line customized from the template (as instructed — the template comment marks this as the "Custom domain system prompt"). It's written here for Monster Hunter World/Iceborne lore: the model is told to act as a specialist on this domain, to answer **only** from retrieved context, and to explicitly refuse (with a fixed fallback sentence) if the answer isn't in the context. The `{context}` placeholder is where the retrieved chunks get inserted at query time.
- `prompt = ChatPromptTemplate.from_messages([...])` — assembles the system prompt and a human message template (`{input}` is replaced with the user's actual question) into the final prompt structure the model will receive.

This cell is also the one we will **modify later for Task 3** (raising `temperature` to `1.0` and deleting the fallback-instruction line), per the lab's instructions.

In [ ]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Your domain is Monster Hunter World and Monster Hunter World: Iceborne lore "
    "— monsters, locations, and story/factions from both the base game and the "
    "Iceborne expansion.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on "
    "the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

### Cell 6 — Pipeline Assembly

This cell wires the retriever (Cell 4) and the prompted LLM (Cell 5) into a single callable RAG chain:

- `from langchain_classic.chains import create_retrieval_chain` — imports the function that assembles a full retrieve-then-generate pipeline. (As of LangChain v1.0, this and the constructor below moved out of the main `langchain` package into the companion `langchain_classic` package, installed in Cell 1.)
- `from langchain_classic.chains.combine_documents import create_stuff_documents_chain` — imports the function that combines retrieved chunks with the prompt in the simplest way: "stuffing" all retrieved chunks directly into the `{context}` slot.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — builds the piece that takes retrieved documents + the prompt template and sends them to the LLM to generate an answer.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` — combines the retriever and the combine-docs chain into one object: given a query, it will first retrieve the top-k relevant chunks, then pass them (plus the query) into the LLM to produce a grounded answer.

After this cell runs, `rag_chain` is the complete, callable chatbot.

In [ ]:
# Cell 6: Pipeline Assembly
# NOTE: as of LangChain v1.0, these constructors moved to the langchain_classic package.
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

### Cell 7 — Testing Your Custom Domain Chatbot

This cell runs the assembled `rag_chain` against the exact in-domain query given in the lab template and prints both the answer and which source chunks were used:

- `user_query = "..."` — the template's generic in-domain test query. It works as-is for any dataset, so it's left unchanged.
- `response = rag_chain.invoke({"input": user_query})` — runs the full pipeline: retrieve top-3 relevant chunks → stuff them into the prompt → call the LLM → return a dict containing the answer and the retrieved context.
- `print(response["answer"])` — prints the model's generated answer.
- The `for` loop over `response["context"]` prints the source filename of each retrieved chunk, so we can verify the answer is actually grounded in our 3 Monster Hunter lore files rather than the model's general knowledge.

In [ ]:
# Cell 7: Testing Your Custom Domain Chatbot
# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
**Summary of the provided documents**

| Item / Item‑type | Ability | Effectiveness | Notes |
|------------------|---------|---------------|-------|
| **Pitfall Trap** | ✔ | ★★☆ (first entry) |  |
| | ✔ | ★★★ (second entry) | A Diablos can run over a pitfall trap when enraged. |
| | ✔ | ★★☆ (third entry) | Used to get the monster to leave an area. |
| **Shock Trap** | ✔ | — | — |
| **Meat** | x | ☆☆☆ | — |

*Other references*

- **Monster Hunter Portable 3rd** – information sourced from mhp3rd.net.  
  - Template examples:  
    - `Uragaan MHP3rd Analysis (Normal)|Normal||True`  
    - `Uragaan MHP3rd Analysis (Broken Chin)|Broken Chin`
- **Monster Hunter World** – “Stun” is represented by **★★★★**.  
- **Frontier** – mentions **Monster Hunter Frontier G9**.

These are the key pieces of data extracted from the documents.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/MHW_Monster_Lore_Complete.txt
Chunk 2 Source: my_data/MHW_Monster_Lore_Complete.tx

In [ ]:
# Bonus Test: Specific In-Domain Query
specific_query = "Describe Nergigante's role and lore in the story of Monster Hunter World."
specific_response = rag_chain.invoke({"input": specific_query})

print("--- SPECIFIC IN-DOMAIN QUERY ANSWER ---")
print(specific_response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(specific_response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- SPECIFIC IN-DOMAIN QUERY ANSWER ---
**Nergigante in the Monster Hunter World story**

- **Elder Dragon status** – Nergigante is one of the rare Elder Dragons that first appeared in *Monster Hunter World*.  
- **Unstoppable ferocity** – It is described as an extremely aggressive monster that continues to fight even if it has to damage itself. The more it is injured or cornered, the more ferocious it becomes.  
- **All‑out aggression** – It will attack anything without hesitation, especially if a creature initiates combat.  
- **Predatory nature toward other Elder Dragons** – Nergigante is particularly ferocious toward other Elder Dragons, actively hunting and feeding on them to gain massive amounts of energy.  
- **Lore speculation** – The lore hints that Nergigante may represent a natural force or phenomenon, with theories suggesting it could be “one of nature’s ways….” (the full speculation is truncated in the provided context).  
- **Variant** – In *Monster Hunter World: Iceborne

---
## Task 1: Domain & Dataset Description

**Domain / problem statement:** This chatbot answers questions strictly grounded in the lore of *Monster Hunter: World* and its expansion *Iceborne* — monster ecology/backstory, region/ecosystem descriptions, and story/faction lore (e.g., the Research Commission, the Fifth Fleet). It is not a gameplay-mechanics assistant (no build guides, weapon stats, or combat tips) — purely narrative/world lore.

**Source filenames uploaded to `./my_data/`:**
1. `MHW_Monster_Lore_Complete.txt` — base-game and Iceborne monster lore, compiled via the Monster Hunter Wiki (Fandom, CC-BY-SA) API
2. `MHW_Location_Lore.txt` — region and ecosystem lore (Ancient Forest, Hoarfrost Reach, Elder's Recess, etc.)
3. `MHW_Story_Faction_Lore.txt` — story, faction, and character lore (Fifth Fleet, Research Commission, The Handler, etc.)


---
## Task 2: Out-of-Domain Fallback Test

Below, we ask a question that is completely outside our dataset — about a monster from a *different* Monster Hunter title (Monster Hunter Rise) that isn't covered by our World/Iceborne-only lore files — to verify the fallback message triggers correctly at the baseline (`temperature=0`) configuration set up in Cell 5.

In [ ]:
# Task 2: Out-of-Domain Fallback Test
ood_query = "Describe Magnamalo's role and lore in Monster Hunter Rise."
ood_response = rag_chain.invoke({"input": ood_query})

print("--- OUT-OF-DOMAIN QUERY ---")
print("Query:", ood_query)
print("\n--- RESPONSE ---")
print(ood_response["answer"])

--- OUT-OF-DOMAIN QUERY ---
Query: Describe Magnamalo's role and lore in Monster Hunter Rise.

--- RESPONSE ---
I cannot answer based on the provided domain data.


---
## Task 3: Hallucination Stress-Test

Per the lab instructions: **"Go to Cell 5. Modify `temperature=0` to `temperature=1.0`. Delete the line in `system_prompt` that reads: 'If the answer cannot be found in the context, reply...'"** — then re-run the query from Task 2 and compare.

The cell below reproduces that exact edit, reassigning the same `llm`, `system_prompt`, `prompt`, `combine_docs_chain`, and `rag_chain` variables from Cells 5–6 (equivalent to editing those cells in place and re-running them), so it mirrors the instructed workflow rather than introducing a separate parallel pipeline.

In [ ]:
# Task 3, Step 1: Re-run Cell 5 with the two required edits
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=1.0  # changed from 0
)

# Fallback-refusal instruction deleted, as instructed
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Your domain is Monster Hunter World and Monster Hunter World: Iceborne lore "
    "— monsters, locations, and story/factions from both the base game and the "
    "Iceborne expansion.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Task 3, Step 1 (cont.): Re-run Cell 6 with the updated llm/prompt
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# Task 3, Step 2: Re-ask the exact same out-of-domain query from Task 2
stress_response = rag_chain.invoke({"input": ood_query})

print("--- STRESS-TEST RESPONSE (temperature=1.0, no fallback instruction) ---")
print("Query:", ood_query)
print("\n--- RESPONSE ---")
print(stress_response["answer"])

--- STRESS-TEST RESPONSE (temperature=1.0, no fallback instruction) ---
Query: Describe Magnamalo's role and lore in Monster Hunter Rise.

--- RESPONSE ---
I’m sorry, but I don’t have enough information in the provided context to describe Magnamalo’s role and lore in Monster Hunter Rise.
